In [ ]:
pip install sentence-transformers torch pandas scikit-learn


# **CONTRADICTION**

In [ ]:
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv("/content/NU.csv")
df["label"] = df["label"].astype(int)  # <-- FIX
sentences1 = df["sentence1"].tolist()
sentences2 = df["sentence2"].tolist()
labels = torch.tensor(df["label"].values, dtype=torch.float)

# Train / test split
s1_train, s1_test, s2_train, s2_test, y_train, y_test = train_test_split(
    sentences1, sentences2, labels, test_size=0.2, random_state=42
)

# Sentence encoder
encoder = SentenceTransformer("all-MiniLM-L6-v2")

# Encode
emb1_train = encoder.encode(s1_train, convert_to_tensor=True)
emb2_train = encoder.encode(s2_train, convert_to_tensor=True)

emb1_test = encoder.encode(s1_test, convert_to_tensor=True)
emb2_test = encoder.encode(s2_test, convert_to_tensor=True)

# Feature: [A, B, |A-B|]
def combine(a, b):
    return torch.cat([a, b, torch.abs(a - b)], dim=1)

X_train = combine(emb1_train, emb2_train)
X_test = combine(emb1_test, emb2_test)

# Simple classifier
class Classifier(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

model = Classifier(X_train.shape[1])
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Training
for epoch in range(30):
    model.train()
    optimizer.zero_grad()
    preds = model(X_train).squeeze()
    loss = criterion(preds, y_train)
    loss.backward()
    optimizer.step()

    if epoch % 5 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

# Evaluation
model.eval()
with torch.no_grad():
    test_preds = model(X_test).squeeze()
    test_preds = (test_preds > 0.5).int()
    acc = (test_preds == y_test.int()).float().mean()

print(f"\nTest accuracy: {acc.item():.2f}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Epoch 0 | Loss: 0.6957
Epoch 5 | Loss: 0.6104
Epoch 10 | Loss: 0.5127
Epoch 15 | Loss: 0.4069
Epoch 20 | Loss: 0.3082
Epoch 25 | Loss: 0.2261

Test accuracy: 0.93


In [ ]:
def predict(a, b):
    ea = encoder.encode(a, convert_to_tensor=True)
    eb = encoder.encode(b, convert_to_tensor=True)
    x = combine(ea.unsqueeze(0), eb.unsqueeze(0))

    prob = model(x).item()
    label = int(prob >= 0.5)

    return prob, label

print(predict(
    "The credit is sufficient",
    "The credit is insufficient"
))  # ~1

print(predict(
    "The credit is sufficient", "The credit is high"
))  # ~0


(0.9160439372062683, 1)
(0.6610137224197388, 1)


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "roberta-large-mnli"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

labels = ["contradiction", "neutral", "entailment"]


config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def predict_contradiction(a, b, threshold=0.7):
    inputs = tokenizer(a, b, return_tensors="pt", truncation=True)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1).squeeze()

    contradiction_prob = probs[0].item()
    label = int(contradiction_prob >= threshold)

    return contradiction_prob, label


In [ ]:
print(predict_contradiction(
    "The credit is sufficient",
    "The credit is insufficient"
))
# (≈0.98, 1)

print(predict_contradiction(
    "The credit is sufficient",
    "The credit is high"
))
# (≈0.10, 0)

print(predict_contradiction(
    "The machine is operational",
    "The machine is out of order"
))
# (≈0.99, 1)


(0.999160885810852, 1)
(0.13610483705997467, 0)
(0.9988388419151306, 1)


In [ ]:
print(predict_contradiction(
    "The product is available",
    "The product is out of stock"
))
# (≈0.98, 1)

print(predict_contradiction(
    "The product is available",
    "The product is available"
))
# (≈0.10, 0)

print(predict_contradiction(
    "The product is available",
    "The product is not available"
))
# (≈0.99, 1)


(0.9944076538085938, 1)
(0.003835658309981227, 0)
(0.9988518953323364, 1)


In [ ]:
df = pd.read_csv("/content/NU.csv")

# siguranță: label numeric
df["label"] = df["label"].astype(int)


In [ ]:
def predict_contradiction(a, b, threshold=0.7):
    inputs = tokenizer(a, b, return_tensors="pt", truncation=True)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1).squeeze()

    contradiction_prob = probs[0].item()  # index 0 = contradiction
    return 1 if contradiction_prob >= threshold else 0  # întoarce 0 sau 1


In [ ]:
from tqdm import tqdm

y_true = df["label"].tolist()
y_pred = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    pred = predict_contradiction(row["sentence1"], row["sentence2"], threshold=0.7)
    y_pred.append(int(pred))  # <--- forțează int



100%|██████████| 143/143 [00:41<00:00,  3.48it/s]


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

acc = accuracy_score(y_true, y_pred)
print("Accuracy:", acc)

print("\nClassification report:")
print(classification_report(
    y_true, y_pred,
    target_names=["NOT contradiction", "contradiction"]
))


Accuracy: 0.951048951048951

Classification report:
                   precision    recall  f1-score   support

NOT contradiction       0.99      0.92      0.95        74
    contradiction       0.92      0.99      0.95        69

         accuracy                           0.95       143
        macro avg       0.95      0.95      0.95       143
     weighted avg       0.95      0.95      0.95       143



In [ ]:
df = pd.read_csv("/content/contradiction_special.csv")

# siguranță: label numeric
df["label"] = df["label"].astype(int)


In [ ]:
def predict_contradiction(a, b, threshold=0.7):
    inputs = tokenizer(a, b, return_tensors="pt", truncation=True)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1).squeeze()

    contradiction_prob = probs[0].item()  # index 0 = contradiction
    return 1 if contradiction_prob >= threshold else 0  # întoarce 0 sau 1


In [ ]:
from tqdm import tqdm

y_true = df["label"].tolist()
y_pred = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    pred = predict_contradiction(row["sentence1"], row["sentence2"], threshold=0.7)
    y_pred.append(int(pred))  # <--- forțează int



100%|██████████| 153/153 [00:45<00:00,  3.33it/s]


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

acc = accuracy_score(y_true, y_pred)
print("Accuracy:", acc)

print("\nClassification report:")
print(classification_report(
    y_true, y_pred,
    target_names=["NOT contradiction", "contradiction"]
))


Accuracy: 0.8758169934640523

Classification report:
                   precision    recall  f1-score   support

NOT contradiction       0.98      0.84      0.90       104
    contradiction       0.73      0.96      0.83        49

         accuracy                           0.88       153
        macro avg       0.86      0.90      0.87       153
     weighted avg       0.90      0.88      0.88       153



# **IMPLICATION **


In [ ]:

import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, classification_report

# ==== 1. Load model ====
model_name = "roberta-large-mnli"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()  # inference mode

# ==== 2. Load CSV dataset ====
df = pd.read_csv("/content/DA.csv")
df["label"] = df["label"].astype(int)

# ==== 3. Prediction function ====
def predict_implication(a, b, threshold=0.7):
    """
    Predicts if sentence1 implies sentence2 using RoBERTa-MNLI
    threshold: probability above which we label as 1 (implication)
    """
    inputs = tokenizer(a, b, return_tensors="pt", truncation=True)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1).squeeze()

    # probs indices in MNLI: [contradiction, neutral, entailment]
    entailment_prob = probs[2].item()  # entailment probability
    return 1 if entailment_prob >= threshold else 0, entailment_prob

# ==== 4. Run predictions ====
y_true = df["label"].tolist()
y_pred = []
probs = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    pred, prob = predict_implication(row["sentence1"], row["sentence2"], threshold=0.7)
    y_pred.append(pred)
    probs.append(prob)

# ==== 5. Evaluation ====
acc = accuracy_score(y_true, y_pred)
print(f"Accuracy: {acc:.4f}\n")
print("Classification report:")
print(classification_report(
    y_true, y_pred, target_names=["Non-implication", "Implication"]
))

# ==== 6. Save results ====
df_result = df.copy()
df_result["y_pred"] = y_pred
df_result["entailment_prob"] = probs
df_result.to_csv("/content/NU_predictions.csv", index=False)
print("\nResults saved to NU_predictions.csv")

# ==== 7. Optional: show mistakes ====
mistakes = df_result[df_result["label"] != df_result["y_pred"]]
print(f"\nNumber of mistakes: {len(mistakes)}")
if len(mistakes) > 0:
    print("\nTop mistakes:")
    print(mistakes[["sentence1", "sentence2", "label", "y_pred", "entailment_prob"]].head(10))


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 98/98 [00:36<00:00,  2.65it/s]

Accuracy: 0.5000

Classification report:
                 precision    recall  f1-score   support

Non-implication       0.48      0.98      0.65        46
    Implication       0.80      0.08      0.14        52

       accuracy                           0.50        98
      macro avg       0.64      0.53      0.39        98
   weighted avg       0.65      0.50      0.38        98


Results saved to NU_predictions.csv

Number of mistakes: 49

Top mistakes:
                                   sentence1  \
5           The machine shows product prices   
46   The user inserts money into the machine   
47                The machine receives money   
48        The user adds money to the machine   
49   The user inserts coins into the machine   
50  The machine receives money from the user   
51                    The user inserts money   
52             The user inserts enough money   
53         The user inserts sufficient money   
54         The machine detects enough credit   

         

In [ ]:
# Instalează dacă nu ai
#!pip install transformers datasets torch --quiet

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# ==== 1. Load CSV ====
df = pd.read_csv("/content/DA_generarat.csv")  # dataset-ul tău implication/neutral
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle complet

# ==== 2. Tokenizer & Dataset HuggingFace ====
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["sentence1"], batch["sentence2"], truncation=True, padding="max_length", max_length=128)

dataset = Dataset.from_pandas(df)
dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# ==== 3. Model ====
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# ==== 4. Training arguments ====
args = TrainingArguments(
    output_dir="/content/implication_vm",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=2,
    eval_strategy="epoch",
    logging_steps=10,
    save_strategy="no",
    seed=42
)

# ==== 5. Trainer ====
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,   # dataset shuffle-uit mai sus
    eval_dataset=dataset
)

# ==== 6. Start training ====
trainer.train()

# ==== 7. Evaluate + save predictions ====
preds_output = trainer.predict(dataset)
probs = torch.softmax(torch.tensor(preds_output.predictions), dim=1)[:,1].tolist()  # probability for implication
y_pred = [1 if p>=0.5 else 0 for p in probs]  # threshold simplu 0.5
y_true = df["label"].tolist()

from sklearn.metrics import accuracy_score, classification_report
acc = accuracy_score(y_true, y_pred)
print(f"\nAccuracy: {acc:.4f}")
print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=["Non-implication","Implication"]))

# Save results with probabilities
df_result = df.copy()
df_result["y_pred"] = y_pred
df_result["implication_prob"] = probs
df_result.to_csv("/content/implication_vm_predictions.csv", index=False)
print("\nPredictions saved to implication_vm_predictions.csv")


Map:   0%|          | 0/84 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument i

Epoch,Training Loss,Validation Loss
1,0.705192,0.681931
2,0.695970,0.677886



Accuracy: 0.5357

Classification report:
                 precision    recall  f1-score   support

Non-implication       0.54      1.00      0.70        45
    Implication       0.00      0.00      0.00        39

       accuracy                           0.54        84
      macro avg       0.27      0.50      0.35        84
   weighted avg       0.29      0.54      0.37        84


Predictions saved to implication_vm_predictions.csv


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
#!pip install transformers datasets torch scikit-learn --quiet

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report

# ==== 1. Load CSV & shuffle ====
df = pd.read_csv("/content/DA_generarat.csv")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle complet

# ==== 2. Tokenizer & Dataset ====
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

dataset = Dataset.from_pandas(df)
dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# ==== 3. Model ====
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# ==== 4. Training arguments ====
args = TrainingArguments(
    output_dir="/content/implication_vm",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    num_train_epochs=8,  # mai multe epoci pentru dataset mic
    eval_strategy="epoch",
    logging_steps=10,
    save_strategy="no",
    seed=42
)

# ==== 5. Trainer ====
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,
    eval_dataset=dataset
)

# ==== 6. Start fine-tuning ====
trainer.train()

# ==== 7. Evaluate ====
preds_output = trainer.predict(dataset)
probs = torch.softmax(torch.tensor(preds_output.predictions), dim=1)[:,1].tolist()  # probability for implication
y_pred = [1 if p>=0.5 else 0 for p in probs]
y_true = df["label"].tolist()

acc = accuracy_score(y_true, y_pred)
print(f"\nAccuracy: {acc:.4f}")
print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=["Non-implication","Implication"]))

# ==== 8. Save predictions ====
df_result = df.copy()
df_result["y_pred"] = y_pred
df_result["implication_prob"] = probs
df_result.to_csv("/content/implication_vm_predictions.csv", index=False)
print("\nPredictions saved to implication_vm_predictions.csv")


Map:   0%|          | 0/84 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument i

Epoch,Training Loss,Validation Loss
1,0.944229,0.648019
2,0.560146,0.437605
3,0.405358,0.561742
4,0.661422,0.522224
5,0.500303,0.181687
6,0.308465,0.335769
7,0.452173,0.150733
8,0.045312,0.143475



Accuracy: 0.9762

Classification report:
                 precision    recall  f1-score   support

Non-implication       0.98      0.98      0.98        45
    Implication       0.97      0.97      0.97        39

       accuracy                           0.98        84
      macro avg       0.98      0.98      0.98        84
   weighted avg       0.98      0.98      0.98        84


Predictions saved to implication_vm_predictions.csv


In [ ]:
# Salvare model + tokenizer
model.save_pretrained("/content/implication_vm_model")
tokenizer.save_pretrained("/content/implication_vm_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/implication_vm_model/tokenizer_config.json',
 '/content/implication_vm_model/tokenizer.json')

In [ ]:

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# ==== 1. Load CSV & shuffle ====
df = pd.read_csv("/content/DA_generarat.csv")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

# ==== 2. K-Fold Cross Validation ====
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
all_acc = []

for train_index, test_index in kf.split(df):
    print(f"\n===== Fold {fold} =====")

    df_train = df.iloc[train_index].reset_index(drop=True)
    df_test = df.iloc[test_index].reset_index(drop=True)

    train_dataset = Dataset.from_pandas(df_train).map(tokenize, batched=True)
    test_dataset = Dataset.from_pandas(df_test).map(tokenize, batched=True)

    # rename label column & set format
    train_dataset = train_dataset.rename_column("label", "labels")
    train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

    test_dataset = test_dataset.rename_column("label", "labels")
    test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

    # ==== 3. Model ====
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    # ==== 4. Training arguments ====
    args = TrainingArguments(
        output_dir=f"./implication_vm_fold{fold}",
        learning_rate=5e-5,
        per_device_train_batch_size=4,
        num_train_epochs=6,  # dataset mic → mai multe epoci
        eval_strategy="epoch",
        logging_steps=10,
        save_strategy="no",
        seed=42
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset
    )

    # ==== 5. Train ====
    trainer.train()

    # ==== 6. Evaluate ====
    preds_output = trainer.predict(test_dataset)
    probs = torch.softmax(torch.tensor(preds_output.predictions), dim=1)[:,1].tolist()
    y_pred = [1 if p >= 0.5 else 0 for p in probs]
    y_true = df_test["label"].tolist()

    acc = accuracy_score(y_true, y_pred)
    all_acc.append(acc)

    print(f"Fold {fold} Accuracy: {acc:.4f}")
    print(classification_report(y_true, y_pred, target_names=["Non-implication","Implication"]))

    fold += 1

print("\n===== K-Fold Results =====")
print(f"Mean Accuracy: {np.mean(all_acc):.4f}")
print(f"Std Dev Accuracy: {np.std(all_acc):.4f}")



===== Fold 1 =====


Map:   0%|          | 0/67 [00:00<?, ? examples/s]

Map:   0%|          | 0/17 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument i

...

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument i

Epoch,Training Loss,Validation Loss
1,0.692142,1.275647
2,0.536540,0.947297
3,0.522688,2.308681
4,0.632379,2.122222
5,0.135884,2.172340
6,0.006694,2.238537


Fold 5 Accuracy: 0.6875
                 precision    recall  f1-score   support

Non-implication       0.57      0.67      0.62         6
    Implication       0.78      0.70      0.74        10

       accuracy                           0.69        16
      macro avg       0.67      0.68      0.68        16
   weighted avg       0.70      0.69      0.69        16


===== K-Fold Results =====
Mean Accuracy: 0.8316
Std Dev Accuracy: 0.0921


In [ ]:
# !pip install transformers datasets torch scikit-learn --quiet

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import os

# ==== 1. Load CSV & shuffle ====
df = pd.read_csv("/content/DA_generarat.csv")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

# ==== 2. K-Fold Cross Validation ====
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
all_acc = []
best_acc = 0.0
best_model_path = "/content/implication_vm_best"

for train_index, test_index in kf.split(df):
    print(f"\n===== Fold {fold} =====")

    df_train = df.iloc[train_index].reset_index(drop=True)
    df_test = df.iloc[test_index].reset_index(drop=True)

    train_dataset = Dataset.from_pandas(df_train).map(tokenize, batched=True)
    test_dataset = Dataset.from_pandas(df_test).map(tokenize, batched=True)

    # rename label column & set format
    train_dataset = train_dataset.rename_column("label", "labels")
    train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

    test_dataset = test_dataset.rename_column("label", "labels")
    test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

    # ==== 3. Model ====
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    # ==== 4. Training arguments ====
    args = TrainingArguments(
        output_dir=f"/content/implication_vm_fold{fold}",
        learning_rate=5e-5,
        per_device_train_batch_size=4,
        num_train_epochs=6,
        eval_strategy="epoch",
        logging_steps=10,
        save_strategy="epoch",
        save_total_limit=1,
        seed=42
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset
    )

    # ==== 5. Train ====
    trainer.train()

    # ==== 6. Evaluate ====
    preds_output = trainer.predict(test_dataset)
    probs = torch.softmax(torch.tensor(preds_output.predictions), dim=1)[:,1].tolist()
    y_pred = [1 if p >= 0.5 else 0 for p in probs]
    y_true = df_test["label"].tolist()

    acc = accuracy_score(y_true, y_pred)
    all_acc.append(acc)

    print(f"Fold {fold} Accuracy: {acc:.4f}")
    print(classification_report(y_true, y_pred, target_names=["Non-implication","Implication"]))

    # ==== 7. Save best model ====
    if acc > best_acc:
        best_acc = acc
        print(f"--> Saving best model from fold {fold} with acc {best_acc:.4f}")
        model.save_pretrained(best_model_path)
        tokenizer.save_pretrained(best_model_path)

    fold += 1

print("\n===== K-Fold Results =====")
print(f"Mean Accuracy: {np.mean(all_acc):.4f}")
print(f"Std Dev Accuracy: {np.std(all_acc):.4f}")

# ==== 8. Load best model and predict on full dataset ====
best_model = AutoModelForSequenceClassification.from_pretrained(best_model_path)
best_tokenizer = AutoTokenizer.from_pretrained(best_model_path)

full_dataset = Dataset.from_pandas(df).map(tokenize, batched=True)
full_dataset = full_dataset.rename_column("label", "labels")
full_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

trainer_best = Trainer(model=best_model)
preds_output = trainer_best.predict(full_dataset)
probs = torch.softmax(torch.tensor(preds_output.predictions), dim=1)[:,1].tolist()
y_pred = [1 if p >= 0.5 else 0 for p in probs]

# ==== 9. Save predictions ====
df_result = df.copy()
df_result["y_pred"] = y_pred
df_result["implication_prob"] = probs
df_result.to_csv("/content/implication_vm_best_predictions.csv", index=False)
print("\nPredictions saved to implication_vm_best_predictions.csv")
print(f"Best model saved to {best_model_path}")



===== Fold 1 =====


Map:   0%|          | 0/67 [00:00<?, ? examples/s]

Map:   0%|          | 0/17 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument i

...

Fold 5 Accuracy: 0.6875
                 precision    recall  f1-score   support

Non-implication       0.57      0.67      0.62         6
    Implication       0.78      0.70      0.74        10

       accuracy                           0.69        16
      macro avg       0.67      0.68      0.68        16
   weighted avg       0.70      0.69      0.69        16


===== K-Fold Results =====
Mean Accuracy: 0.8434
Std Dev Accuracy: 0.0864


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/84 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



Predictions saved to implication_vm_best_predictions.csv
Best model saved to /content/implication_vm_best


In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, classification_report

# ==== 1. Încarcă modelul și tokenizerul salvat ====
model_path = "/content/implication_vm_best"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()  # important să nu fie în mod training

# ==== 2. Încarcă dataset-ul complet ====
df = pd.read_csv("/content/DA_generarat.csv")  # același CSV pe care l-ai folosit la K-Fold

# ==== 3. Predict ====
y_true = df["label"].tolist()
y_pred = []
implication_probs = []

for _, row in df.iterrows():
    inputs = tokenizer(
        row["sentence1"], row["sentence2"],
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    )
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1).squeeze()
        implication_prob = probs[1].item()  # index 1 = implication
        implication_probs.append(implication_prob)
        y_pred.append(1 if implication_prob >= 0.5 else 0)  # threshold 0.5

# ==== 4. Evaluați acuratețea și classification report ====
acc = accuracy_score(y_true, y_pred)
print(f"\nAccuracy: {acc:.4f}\n")
print(classification_report(y_true, y_pred, target_names=["Non-implication","Implication"]))

# ==== 5. Salvează predicțiile într-un CSV ====
df_result = df.copy()
df_result["y_pred"] = y_pred
df_result["implication_prob"] = implication_probs
df_result.to_csv("/content/implication_vm_best_predictions.csv", index=False)
print("Predictions saved to implication_vm_best_predictions.csv")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Accuracy: 0.9762

                 precision    recall  f1-score   support

Non-implication       0.98      0.98      0.98        45
    Implication       0.97      0.97      0.97        39

       accuracy                           0.98        84
      macro avg       0.98      0.98      0.98        84
   weighted avg       0.98      0.98      0.98        84

Predictions saved to implication_vm_best_predictions.csv


# **REDUDANCY**

In [ ]:
# ===============================
# Test Redundancy cu RoBERTa MNLI
# ===============================

import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report

# --- Setup model ---
model_name = "roberta-large-mnli"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# --- Functie pentru predict redundancy ---
# 1 = redundant (entailment), 0 = neutral
def predict_redundancy(a, b, threshold=0.7):
    inputs = tokenizer(a, b, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1).squeeze()
    entailment_prob = probs[2].item()  # index 2 = entailment
    return 1 if entailment_prob >= threshold else 0

# --- Citire CSV ---
csv_path = "/content/redudancy.csv"  # schimba cu path-ul tau
df = pd.read_csv(csv_path)

# asigura-te ca label este numeric
df["label"] = df["label"].astype(int)

# --- Predict pentru fiecare rand ---
y_true = df["label"].tolist()
y_pred = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    pred = predict_redundancy(row["sentence1"], row["sentence2"], threshold=0.7)
    y_pred.append(pred)

# --- Evaluare ---
acc = accuracy_score(y_true, y_pred)
print("Accuracy:", acc)

print("\nClassification report:")
print(classification_report(
    y_true, y_pred,
    target_names=["neutral", "redundant"]
))

# --- Optional: vezi si probabilitatile pentru fiecare clasa ---
def predict_probs(a, b):
    inputs = tokenizer(a, b, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1).squeeze()
    return {
        "contradiction": probs[0].item(),
        "neutral": probs[1].item(),
        "entailment": probs[2].item()
    }

# Exemplu:
ex = df.iloc[0]
print("\nProbabilities example:")
print(ex["sentence1"])
print(ex["sentence2"])
print(predict_probs(ex["sentence1"], ex["sentence2"]))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 116/116 [01:11<00:00,  1.63it/s]


Accuracy: 0.8879310344827587

Classification report:
              precision    recall  f1-score   support

     neutral       0.92      0.91      0.91        74
   redundant       0.84      0.86      0.85        42

    accuracy                           0.89       116
   macro avg       0.88      0.88      0.88       116
weighted avg       0.89      0.89      0.89       116


Probabilities example:
The machine is idle
The machine is not doing anything
{'contradiction': 0.0009900759905576706, 'neutral': 0.011012773960828781, 'entailment': 0.9879971742630005}


In [ ]:
# ===============================
# Test Redundancy cu RoBERTa MNLI
# ===============================

import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report

# --- Setup model ---
model_name = "roberta-large-mnli"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# --- Functie pentru predict redundancy ---
# 1 = redundant (entailment), 0 = neutral
def predict_redundancy(a, b, threshold=0.7):
    inputs = tokenizer(a, b, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1).squeeze()
    entailment_prob = probs[2].item()  # index 2 = entailment
    return 1 if entailment_prob >= threshold else 0

# --- Citire CSV ---
csv_path = "/content/redudancy.csv"  # schimba cu path-ul tau
df = pd.read_csv(csv_path)

# asigura-te ca label este numeric
df["label"] = df["label"].astype(int)

# --- Predict pentru fiecare rand ---
y_true = df["label"].tolist()
y_pred = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    pred = predict_redundancy(row["sentence1"], row["sentence2"], threshold=0.6)
    y_pred.append(pred)

# --- Evaluare ---
acc = accuracy_score(y_true, y_pred)
print("Accuracy:", acc)

print("\nClassification report:")
print(classification_report(
    y_true, y_pred,
    target_names=["neutral", "redundant"]
))

# --- Optional: vezi si probabilitatile pentru fiecare clasa ---
def predict_probs(a, b):
    inputs = tokenizer(a, b, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1).squeeze()
    return {
        "contradiction": probs[0].item(),
        "neutral": probs[1].item(),
        "entailment": probs[2].item()
    }

# Exemplu:
ex = df.iloc[0]
print("\nProbabilities example:")
print(ex["sentence1"])
print(ex["sentence2"])
print(predict_probs(ex["sentence1"], ex["sentence2"]))


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 116/116 [01:01<00:00,  1.89it/s]


Accuracy: 0.8879310344827587

Classification report:
              precision    recall  f1-score   support

     neutral       0.93      0.89      0.91        74
   redundant       0.82      0.88      0.85        42

    accuracy                           0.89       116
   macro avg       0.88      0.89      0.88       116
weighted avg       0.89      0.89      0.89       116


Probabilities example:
The machine is idle
The machine is not doing anything
{'contradiction': 0.0009900759905576706, 'neutral': 0.011012773960828781, 'entailment': 0.9879971742630005}


# **EXCLUSION**

In [ ]:
import csv
from transformers import pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix,
    classification_report
)

MODEL_NAME = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
CSV_PATH = "/content/exclusion.csv"
THRESHOLD = 0.5

clf = pipeline("text-classification", model=MODEL_NAME, top_k=None)

def normalize_nli_label(lbl):
    lbl = lbl.lower()
    if "contrad" in lbl or lbl == "label_0":
        return "contradiction"
    if "neutral" in lbl or lbl == "label_1":
        return "neutral"
    if "entail" in lbl or lbl == "label_2":
        return "entailment"

def load_csv(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        for r in reader:
            if len(r) < 3:
                continue
            s1, s2 = r[0].strip(), r[1].strip()
            y = 1 if r[2].strip() == "1" else 0
            rows.append((s1, s2, y))
    return rows

def predict_prob_contradiction(premise, hypothesis):
    out = clf({"text": premise, "text_pair": hypothesis})
    probs = {}
    for x in out:
        probs[normalize_nli_label(x["label"])] = float(x["score"])
    return probs.get("contradiction", 0.0)

# ================= RUN =================

data = load_csv(CSV_PATH)

y_true = []
y_pred = []

for s1, s2, y in data:
    p_contra = predict_prob_contradiction(s1, s2)
    pred = 1 if p_contra >= THRESHOLD else 0
    y_true.append(y)
    y_pred.append(pred)

# ================= METRICS =================

acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
f2 = fbeta_score(y_true, y_pred, beta=2)

cm = confusion_matrix(y_true, y_pred)
report = classification_report(y_true, y_pred, digits=4)

print("\n=== Metrics ===")
print(f"Accuracy : {acc*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall   : {recall*100:.2f}%")
print(f"F1       : {f1*100:.2f}%")
print(f"F2       : {f2*100:.2f}%")

print("\nConfusion matrix:")
print(cm)

print("\nClassification report:")
print(report)


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== Metrics ===
Accuracy : 87.96%
Precision: 92.00%
Recall   : 67.65%
F1       : 77.97%
F2       : 71.43%

Confusion matrix:
[[72  2]
 [11 23]]

Classification report:
              precision    recall  f1-score   support

           0     0.8675    0.9730    0.9172        74
           1     0.9200    0.6765    0.7797        34

    accuracy                         0.8796       108
   macro avg     0.8937    0.8247    0.8484       108
weighted avg     0.8840    0.8796    0.8739       108

